# UnifyWeaver의 고급 재귀 패턴

이 노트북은 UnifyWeaver가 감지하고 최적화할 수 있는 4가지 주요 재귀 패턴을 보여줍니다:

1. **꼬리 재귀 (Tail Recursion)** - 누산기를 사용한 반복 루프
2. **선형 재귀 (Linear Recursion)** - 메모화가 적용된 단일 재귀 호출
3. **트리 재귀 (Tree Recursion)** - 구조의 여러 부분에 대한 다중 재귀 호출
4. **상호 재귀 (Mutual Recursion)** - 서술어들이 순환 구조로 서로를 호출

## 학습 목표

- 다양한 재귀 패턴 이해
- UnifyWeaver가 각 패턴을 감지하고 최적화하는 방법 확인
- 성능 특성 비교
- 각 패턴을 언제 사용해야 하는지 학습

## 설정

UnifyWeaver 환경을 초기화합니다.

In [ ]:
% 초기화 로드
['../init'].

% 필요한 모듈 로드
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## 패턴 1: 꼬리 재귀 (Tail Recursion)

꼬리 재귀는 누산기(accumulator)를 사용하여 중간 결과를 전달하며, 재귀 호출이 함수의 **마지막 동작**입니다.

### 예제: 리스트의 항목 수 세기

In [ ]:
% 꼬리 재귀 count_items 정의
:- dynamic count_items/3.

% 기저 사례: 빈 리스트, 누산기 반환
count_items([], Acc, Acc).

% 재귀 사례: 누산기 증가 후 꼬리에 대해 재귀 호출
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← 꼬리 위치!

### Prolog에서 테스트

In [ ]:
% 테스트: [a,b,c,d,e]의 항목 수 세기
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### 패턴 감지 확인

In [ ]:
% 꼬리 재귀로 감지되었는지 확인
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Bash로 컴파일

In [ ]:
% 컴파일 및 저장
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "[a,b,c,d,e]의 항목 수 세는 중:"
count_items "[a,b,c,d,e]" 0 ""

## 패턴 2: 선형 재귀 (Linear Recursion)

선형 재귀는 절당 **정확히 하나의** 재귀 호출을 가지며, 재귀 호출이 반환된 후에 추가 계산이 수행됩니다.

### 예제: 팩토리얼 (Factorial)

In [ ]:
% 팩토리얼 정의
:- dynamic factorial/2.

% 기저 사례
factorial(0, 1).

% 재귀 사례: 정확히 하나의 재귀 호출
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← 하나의 재귀 호출
    F is N * F1.        % ← 호출 후 계산

### Prolog에서 테스트

In [ ]:
% 테스트: 5의 팩토리얼
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### 패턴 감지 확인

In [ ]:
% 선형 재귀로 감지되었는지 확인
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Bash로 컴파일

In [ ]:
% 컴파일 및 저장
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % 함수 정의만 유지 (Brush는 source된 스크립트를 직접 실행으로 처리함)
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "5의 팩토리얼:"
factorial 5 ""
echo ""
echo "10의 팩토리얼:"
factorial 10 ""

## 패턴 3: 트리 재귀 (Tree Recursion)

트리 재귀는 구조의 서로 다른 부분을 처리하기 위해 **여러 번의** 재귀 호출을 수행합니다.

### 예제: 트리 합계 (Tree Sum)

In [ ]:
% 이진 트리를 위한 tree_sum 정의
% 트리 형식: [값, 왼쪽서브트리, 오른쪽서브트리] 또는 []
:- dynamic tree_sum/2.

% 기저 사례: 빈 트리의 합은 0
tree_sum([], 0).

% 재귀 사례: 합계 = 값 + 왼쪽합 + 오른쪽합
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← 첫 번째 재귀 호출
    tree_sum(R, RS),   % ← 두 번째 재귀 호출
    Sum is V + LS + RS.

### Prolog에서 테스트

In [ ]:
% 테스트: [5, [3, [1, [], []], []], [2, [], []]]의 tree_sum
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Bash로 컴파일

In [ ]:
% 컴파일 및 저장
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "[5,[3,[1,[],[]],[]],[2,[],[]]]의 트리 합계:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## 패턴 4: 상호 재귀 (Mutual Recursion)

상호 재귀는 둘 이상의 서술어가 순환하여 서로를 호출할 때 발생합니다.

### 예제: 짝수(Even)와 홀수(Odd)

In [ ]:
% 상호 재귀 is_even 및 is_odd 정의
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even 기저 사례
is_even(0).

% is_even 재귀: N-1이 홀수이면 N은 짝수
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← is_odd 호출

% is_odd 기저 사례
is_odd(1).

% is_odd 재귀: N-1이 짝수이면 N은 홀수
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← is_even 호출

### Prolog에서 테스트

In [ ]:
% 짝수/홀수 테스트
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### 상호 재귀 확인

In [ ]:
% 호출 그래프 작성 및 SCC 검색
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Bash로 컴파일

In [ ]:
% 상호 재귀 그룹 컴파일
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### 생성된 Bash 테스트

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "is_even 및 is_odd 테스트 중:"
is_even 0 >/dev/null && echo "✓ 0은 짝수"
is_even 4 >/dev/null && echo "✓ 4는 짝수"
is_odd 3 >/dev/null && echo "✓ 3은 홀수"
is_odd 7 >/dev/null && echo "✓ 7은 홀수"
is_even 5 >/dev/null 2>&1 || echo "✓ 5는 짝수가 아님"

## 패턴 비교

각 패턴의 특성을 비교해 보겠습니다:

| 패턴 | 재귀 호출 | 최적화 방식 | 공간 복잡도 | 적합한 용도 |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **꼬리 재귀** | 1회 (꼬리 위치) | 반복 루프 | O(1) | 누산기, 선형 스캔 |
| **선형 재귀** | 1회 (임의 위치) | Fold + 메모화 | O(n) 메모 테이블 | 피보나치, 팩토리얼 |
| **트리 재귀** | 2회 이상 (구조 각 부분) | 구조적 분해 | O(depth) 스택 | 트리/그래프 연산 |
| **상호 재귀** | 1회 이상 (서술어 간) | 공유 메모화 | O(n) 공유 테이블 | 짝수/홀수, 상호 정의 |

## 패턴 감지 순서

UnifyWeaver는 다음 순서로 패턴 일치를 시도합니다:

1. **꼬리 재귀** (가장 효율적)
2. **선형 재귀** (금지되지 않은 경우)
3. **트리 재귀** (구조적)
4. **상호 재귀** (강결합 컴포넌트 SCC 감지)
5. **기본 재귀** (대체 폴백)

`forbid_linear_recursion/1`을 사용하여 감지 동작을 제어할 수 있습니다.

## 연습 과제: 직접 해보세요!

다음 서술어를 정의하고 컴파일해 보세요:

### 1. 꼬리 재귀 합계
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. 선형 재귀 피보나치
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. 트리 높이
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% 여기에 코드를 작성하세요!


## 요약

이 노트북에서 배운 내용:

✅ UnifyWeaver의 4가지 주요 재귀 패턴

✅ Prolog에서 각 패턴을 정의하는 방법

✅ UnifyWeaver가 각 패턴을 감지하고 최적화하는 방법

✅ 각 패턴의 성능 특성

✅ 각 패턴을 언제 사용해야 하는지

## 다음 단계

고급 코드 분석 및 시각화에 대해 배우려면 **노트북 3: 호출 그래프 시각화**로 이동하세요!